# SCP Step 5 - Simulate (Colab)

This notebook can bootstrap SCP on Colab, then uses the same `SimulationSession` backend as the local notebook.


In [ ]:
# Colab/Linux bootstrap: locate or clone SCP, install deps, and set repo root
import os
import sys
import subprocess
import importlib
from pathlib import Path


def _is_scp_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "cells").is_dir()
        and (path / "modules").is_dir()
        and (path / "run_pipeline.py").is_file()
    )


def _find_scp_root(start: Path):
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if _is_scp_root(cand):
            return cand
    try:
        for child in start.iterdir():
            if child.is_dir() and _is_scp_root(child):
                return child.resolve()
    except Exception:
        pass
    return None


def _repo_url_with_token(repo_url: str) -> str:
    token = (
        os.environ.get("SCP_GIT_TOKEN")
        or os.environ.get("SCP_GITHUB_TOKEN")
        or os.environ.get("GITHUB_TOKEN")
    )
    if token and repo_url.startswith("https://") and "@" not in repo_url:
        return repo_url.replace("https://", f"https://{token}@", 1)
    return repo_url


def _redact_repo_url(url: str) -> str:
    if "://" not in url or "@" not in url:
        return url
    scheme, rest = url.split("://", 1)
    host_part = rest.split("@", 1)[1]
    return f"{scheme}://***@{host_part}"


def _clone_repo(repo_url: str, repo_dir: Path, branch=None) -> Path:
    repo_dir = repo_dir.expanduser().resolve()
    if _is_scp_root(repo_dir):
        return repo_dir

    if repo_dir.exists() and any(repo_dir.iterdir()):
        raise FileExistsError(
            f"Repo dir exists but is not an SCP checkout: {repo_dir}. "
            "Set SCP_REPO_DIR to an empty folder or existing SCP clone."
        )

    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    auth_url = _repo_url_with_token(repo_url)

    cmd = ["git", "clone", "--depth", "1"]
    if branch:
        cmd += ["--branch", branch]
    cmd += [auth_url, str(repo_dir)]

    shown_cmd = [*cmd[:-2], _redact_repo_url(auth_url), str(repo_dir)]
    print("Cloning SCP repo:", " ".join(shown_cmd))

    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Failed to clone SCP repo. If this repo is private, set SCP_GIT_TOKEN (or SCP_GITHUB_TOKEN/GITHUB_TOKEN) in Colab secrets/environment."
        ) from exc

    if not _is_scp_root(repo_dir):
        raise RuntimeError(f"Clone completed but repo layout is invalid: {repo_dir}")
    return repo_dir


def _ensure_python_pkg(import_name: str, pip_name=None) -> None:
    try:
        importlib.import_module(import_name)
        return
    except Exception:
        pass
    pkg = pip_name or import_name
    print(f"Installing missing package: {pkg}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])


IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
AUTO_CLONE_REPO = os.environ.get("SCP_AUTO_CLONE", "1") not in ("0", "false", "False")
DEFAULT_REPO_URL = "https://github.com/HunterBushnell/SCP.git"
REPO_URL = os.environ.get("SCP_REPO_URL", DEFAULT_REPO_URL)
REPO_BRANCH = os.environ.get("SCP_REPO_BRANCH", "") or None
DEFAULT_REPO_DIR = Path("/content/SCP") if IN_COLAB else (Path.cwd() / "SCP")
REPO_DIR = Path(os.environ.get("SCP_REPO_DIR", str(DEFAULT_REPO_DIR)))

repo_root = _find_scp_root(Path.cwd())
cloned = False
if repo_root is None and AUTO_CLONE_REPO:
    repo_root = _clone_repo(REPO_URL, REPO_DIR, branch=REPO_BRANCH)
    cloned = True

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate SCP repo from current directory. "
        "Set SCP_REPO_DIR/SCP_REPO_URL or run notebook from inside SCP."
    )

repo_root = repo_root.resolve()
os.environ["SCP_ROOT"] = str(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("SCP root:", repo_root)
from modules.notebook_helpers import ensure_scp_repo_on_syspath
repo_root = ensure_scp_repo_on_syspath(repo_root)
os.environ["SCP_ROOT"] = str(repo_root)
if cloned:
    print("SCP was cloned for this session.")

# Only needed on fresh Colab sessions
INSTALL_DEPS = IN_COLAB
if INSTALL_DEPS:
    _ensure_python_pkg("numpy")
    _ensure_python_pkg("pandas")
    _ensure_python_pkg("matplotlib")
    _ensure_python_pkg("scipy")
    _ensure_python_pkg("h5py")
    _ensure_python_pkg("neuron")
    # Required by modules.load_cell used in Step 5
    _ensure_python_pkg("allensdk")

    # Needed to compile NEURON modfiles when x86_64 binaries are absent
    subprocess.check_call(["apt-get", "update"])
    subprocess.check_call(["apt-get", "install", "-y", "build-essential"])

# Required external inputs (print if missing)
required_external = [
    repo_root / "external_data" / "pyrFiringRateAvg.csv",
    repo_root / "external_data" / "PV_1000tr.pkl",
    repo_root / "external_data" / "SST_1000tr.pkl",
]
missing = [p for p in required_external if not p.is_file()]
if missing:
    print("Missing external inputs:")
    for p in missing:
        print(" -", p)


def ensure_modfiles(tune_dir: Path, compile_modfiles: bool = True) -> None:
    mod_dir = tune_dir / "modfiles"
    dll_candidates = [
        mod_dir / "x86_64" / ".libs" / "libnrnmech.so",
        mod_dir / "x86_64" / "libnrnmech.so",
    ]
    if any(p.is_file() for p in dll_candidates):
        return
    if not mod_dir.is_dir():
        print(f"Missing modfiles dir: {mod_dir}")
        return
    if compile_modfiles:
        print("Compiling modfiles with nrnivmodl...")
        subprocess.check_call(["nrnivmodl"], cwd=str(mod_dir))
    else:
        print("Missing compiled modfiles; run nrnivmodl in", mod_dir)


## Quick Start

Run the bootstrap cell first. After that, the workflow is the same as the local Step 5 notebook: choose a cell/tune, prepare the session, run, and inspect minimal diagnostics.


In [ ]:
# Shared imports for local and Colab Step 5 workflows
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from modules.simulation import SimulationOptions, SimulationSession
from modules import run_sim

repo_root = Path(os.environ.get("SCP_ROOT", repo_root)).resolve()
print("SCP repo:", repo_root)


In [ ]:
# User settings: choose the model and optional run overrides
cell_name = "SST"      # bundled examples: "PV" or "SST"
tune_name = "seg_tuned"

tune_dir = repo_root / "cells" / cell_name / "tunes" / tune_name
if not tune_dir.is_dir():
    raise FileNotFoundError(f"Tune directory not found: {tune_dir}")

# Keep these as None/False to use cell_configs/sim_config.json directly.
run_mode = None              # None, "single", or "multi"
n_trials = None              # e.g. 1 for a quick smoke run
seed = None                  # e.g. 1234 for reproducibility
run_iclamp = False           # True for current injection instead of synaptic inputs
force_save = False           # True to save even if sim_config disables saving
output_stem = None           # e.g. "notebook_test"; None uses sim_config/defaults
output_dir = None            # None -> tune_dir/output_data
preview_synapses = False     # True for a lightweight synapse-placement preview

options = SimulationOptions(
    mode=run_mode,
    n_trials=n_trials,
    seed=seed,
    iclamp=run_iclamp,
    force_save=force_save,
    output_stem=output_stem,
    output_dir=output_dir,
)

print("Tune dir:", tune_dir)


In [ ]:
# Prepare the simulation session
# Colab bootstrap defines ensure_modfiles(); local notebooks can ignore it.
try:
    ensure_modfiles(tune_dir, compile_modfiles=IN_COLAB)
except NameError:
    pass

session = SimulationSession.from_tune(tune_dir, options=options)
session.prepare()

print("Session summary:")
for key, value in session.summary().items():
    print(f"  {key}: {value}")

if session.groups_cfg:
    print("Synapse groups:", ", ".join(session.groups_cfg.keys()))
else:
    print("Synapse groups: none (IClamp mode or no active groups)")


In [ ]:
# Optional: preview synapse placement without running the simulation
if preview_synapses and not session.iclamp_enabled:
    syn_state = session.preview_synapses(trial_idx=0)
    print("Synapse preview complete.")
    for group_name, records in syn_state.get("records", {}).items():
        print(f"  {group_name}: {len(records)} synapses")
elif preview_synapses:
    print("Skipping synapse preview because IClamp mode is enabled.")
else:
    print("Synapse preview disabled.")


In [ ]:
# Run the simulation and save according to sim_config/options
results = session.run()
saved_path = session.save()

if saved_path is None:
    print("Results not saved. Enable saving in sim_config or set force_save=True.")
else:
    print("Results saved to:", saved_path)


In [ ]:
# Minimal diagnostics: spike counts and Vm trace
def _spike_counts(spikes):
    if spikes is None:
        return []
    if isinstance(spikes, list):
        return [len(np.asarray(trial)) for trial in spikes]
    return [len(np.asarray(spikes))]

mode = results.get("mode")
spike_counts = _spike_counts(results.get("spikes"))
if spike_counts:
    print("Mode:", mode)
    print("Spike counts:", spike_counts)
    print("Mean spikes/trial:", float(np.mean(spike_counts)))
else:
    print("Mode:", mode)
    print("No spike vector found; this is expected for some IClamp runs.")

traces = results.get("traces", {}) or {}
T = traces.get("T")
V = traces.get("V")

if T is not None and V is not None:
    plt.figure(figsize=(8, 4))
    if isinstance(V, list):
        for idx, trace in enumerate(V[:3]):
            plt.plot(T, trace, lw=1.0, label=f"trial {idx}")
        if len(V) > 1:
            plt.legend()
    else:
        plt.plot(T, V, lw=1.2)
    plt.xlabel("Time (ms)")
    plt.ylabel("Vm (mV)")
    plt.title(f"{cell_name} / {tune_name} ({mode})")
    plt.tight_layout()
else:
    print("No Vm trace stored. Increase n_traces_to_save or cell_recording settings if needed.")


## Optional Utilities

Use this only when you want to inspect a previous saved run from disk. More detailed analysis remains in `6_analysis.ipynb`.


In [ ]:
# Optional: load a previous saved run
load_previous = False
previous_run = tune_dir / "output_data" / "<run_name>"  # folder or run_manifest.json

if load_previous:
    results = run_sim.load_results(previous_run)
    print("Loaded:", previous_run)
    print("Mode:", results.get("mode"))
